## Ноутбук 02: EDA и предобработка отзывов пользователей (Reviews)
### Цели и задачи исследования:
1. **Очистка данных и валидация типов:** Проверка пропусков, дубликатов, обработка временных полей (`timestamp_created` -> `datetime`) и валидация структурных аномалий.
2. **Анализ распределения взаимодействий (EDA):** Исследование активности пользователей (User Degree), популярности игр (Item Degree) и общей разреженности графа взаимодействий (Sparsity).
3. **Формирование сигнала увлеченности:** Анализ распределения времени игры (`playtime_forever`) и его логарифмирование ($\log(1 + \text{playtime})$) для создания взвешенного фидбека.
4. **Сохранение подготовленного датасета:** Экспорт очищенной базовой таблицы в формат `reviews_clean.parquet`.

In [1]:
import polars as pl
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

pl.Config.set_tbl_cols(-1)

data_dir = '../data'
output_dir = '../data/processed'

### Загрузка и общая структура данных

In [2]:
reviews_df = pl.read_csv(
    os.path.join(data_dir, 'reviews.csv'),
    ignore_errors=True
)

reviews_df.head(3)

recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review_text,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
i64,i64,i64,i64,i64,i64,i64,f64,i64,str,str,i64,i64,bool,i64,i64,f64,i64,bool,bool,bool,str,str
10000000,264220,76561198085405844,760,74,12,0,12.0,1399059573,"""polish""","""What's a crap. This game costs…",1399059965,1399059965,true,0,1,0.459906,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"
100001066,1006440,76561198014439859,485,234,424,0,424.0,1632666417,"""russian""","""Игра в жанре квеста point-&-cl…",1632673992,1632673992,true,8,0,0.63077,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"
100002344,320721,76561198048038590,0,385,0,0,null,0,"""german""","""Erneut gibt es einen DLC mit d…",1632675631,1632675631,false,1,0,0.52381,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 00:47:55.754043+00:…"


In [3]:
reviews_df.glimpse()

Rows: 1048148
Columns: 23
$ recommendationid                <i64> 10000000, 100001066, 100002344, 100002361, 100002504, 100002591, 100003580, 100003676, 100003804, 100005000
$ appid                           <i64> 264220, 1006440, 320721, 1604700, 1338560, 1418320, 1232500, 726870, 1721670, 1638870
$ author_steamid                  <i64> 76561198085405844, 76561198014439859, 76561198048038590, 76561197994386273, 76561198138996331, 76561199208288640, 76561199089236340, 76561198155272674, 76561198056175175, 76561199059802534
$ author_num_games_owned          <i64> 760, 485, 0, 0, 816, 0, 0, 0, 315, 0
$ author_num_reviews              <i64> 74, 234, 385, 3, 26, 1, 2, 13, 14, 4
$ author_playtime_forever         <i64> 12, 424, 0, 86, 25, 60, 311, 639, 0, 12
$ author_playtime_last_two_weeks  <i64> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
$ author_playtime_at_review       <f64> 12.0, 424.0, null, 86.0, 25.0, 60.0, 261.0, 188.0, null, 12.0
$ author_last_played              <i64> 1399059573, 1632666417, 0,

In [4]:
reviews_df.null_count()

recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review_text,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,98148,98148,178857,98148,0,769,0,0,0,0,0,0,0,0,0,0,0,0


### Качество данных

Проверка на дубликаты:

In [5]:
full_duplicates = reviews_df.is_duplicated().sum()
print(f"Полных дубликатов: {full_duplicates}")

user_item_duplicates = (reviews_df
    .group_by(['author_steamid', 'appid'])
    .len()
    .filter(pl.col('len') > 1)
)
print(f"Уникальных user-item пар с повторениями: {len(user_item_duplicates)}")
print(f"Всего лишних записей в повторных user-item парах: {(user_item_duplicates['len'] - 1).sum()}")

invalid_user_ids = reviews_df.filter(
    pl.col("author_steamid").is_null() |
    (pl.col("author_steamid") <= 0)
)
invalid_game_ids = reviews_df.filter(
    pl.col("appid").is_null() |
    (pl.col("appid") <= 0)
)
print(f"Некорректных user_id: {len(invalid_user_ids)}")
print(f"Некорректных appid: {len(invalid_game_ids)}")


invalid_playtime = reviews_df.filter(
    pl.col("author_playtime_forever").is_null() |
    (pl.col("author_playtime_forever") < 0)
)
print(f"Некорректных значений playtime: {len(invalid_playtime)}")

Полных дубликатов: 0
Уникальных user-item пар с повторениями: 6
Всего лишних записей в повторных user-item парах: 8
Некорректных user_id: 0
Некорректных appid: 0
Некорректных значений playtime: 98148


Проанализируем user-item дубликаты:

In [6]:
duplicates = reviews_df.join(user_item_duplicates, on=['author_steamid', 'appid'], how='semi')
duplicates

recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review_text,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
i64,i64,i64,i64,i64,i64,i64,f64,i64,str,str,i64,i64,bool,i64,i64,f64,i64,bool,bool,bool,str,str
10343695,207000,76561198134172548,50,1,68,0,68.0,1402144291,"""russian""","""да игра хорошая но разовая""",1401883692,1401883692,true,0,0,0.5,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 00:57:00.523312+00:…"
10343696,207000,76561198134172548,50,1,68,0,68.0,1402144291,"""russian""","""да игра хорошая но разовая""",1401883692,1401883692,true,0,0,0.5,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 00:57:00.523312+00:…"
10345310,253370,76561198069244336,0,1,68,0,64.0,1402495673,"""english""","""Great for beginners in designi…",1401895809,1401895809,true,11,0,0.513139,3,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 00:57:00.523312+00:…"
10345311,253370,76561198069244336,0,1,68,0,64.0,1402495673,"""english""","""Great for beginners in designi…",1401895809,1401895809,true,12,0,0.488208,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 00:57:00.523312+00:…"
10355746,263480,76561198050995086,58,1,69,0,65.0,1405220390,"""spanish""","""not recomended at all. han ten…",1401974097,1401974097,false,2,0,0.477738,0,true,false,true,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 00:57:00.523312+00:…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
116635839,896230,76561198087296603,0,6,10873,0,2761.0,1696024466,"""english""","""I'm still playing for some ine…",1654616002,1654616018,false,3,0,0.5625,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-09 01:01:54.538460+00:…"
9469713,251510,76561197971290952,602,8,null,null,182.0,null,"""english""","""Only on the second set of leve…",1394827527,1394827527,true,4,0,0.502895,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 03:09:34.083959+00:…"
9469714,251510,76561197971290952,602,8,null,null,182.0,null,"""english""","""Only on the second set of leve…",1394827527,1394827527,true,0,0,0.5,0,true,false,false,"""2025-09-07 12:51:00.564782+00:…","""2025-09-08 03:09:34.083959+00:…"


Проверим, у встречаются ли разные `timestamp_created` у дубликатов:

In [7]:
duplicates.group_by(['author_steamid', 'appid']).agg(pl.col('timestamp_created').n_unique())

author_steamid,appid,timestamp_created
i64,i64,u32
76561198087296603,896230,2
76561198134172548,207000,1
76561198069244336,253370,1
76561197961949171,1336840,2
76561197971290952,251510,1
76561198050995086,263480,1


Некоторые дубликаты отличаются не только recommendationid, но и количеством лайков (`votes_up`), при чем 

In [ ]:
reviews_clean = (
    reviews_df
    .sort(['author_steamid', 'appid', 'timestamp_created', 'votes_up'], descending=[False, False, True, True])
    .unique(subset=['author_steamid', 'appid'], keep='first')
)

assert len(reviews_clean) == len(reviews_clean.unique(subset=['author_steamid', 'appid']))
print(f"Очистка успешна! Осталось {len(reviews_clean):,} записей.")

Очистка успешна! Осталось 1,048,140 записей.
